# 👋 AutoGluon Regression Tutorial for Prediction

Last updated: 03 Sep 2025

AutoGluon is an open-source, automated machine learning library in Python that simplifies building and deploying machine learning models. It provides a low-code interface for regression, classification, and time-series forecasting, ideal for researchers and citizen data scientists. AutoGluon automates data preprocessing, model selection, hyperparameter tuning, and ensemble creation, delivering high performance with minimal code.

This notebook analyzes prediction in the Chilwa Basin using the dataset from August 2025. It follows the workflow: **Setup** ➡️ **Data Analysis** ➡️ **Train Models** ➡️ **Analyze Model** ➡️ **Visualize Results** ➡️ **Save Outputs**. Results are formatted for a scientific paper.

**Dataset**: Chilwa Basin Dataset (starting from Jan 1, 1946, and extending into 2025, with future updates expected), containing environmental and health data.
**Objective**: Predict a user-specified target using user-selected environmental or health features.


# 🚧 Installation

Install AutoGluon, graphviz, and dependencies. Run this cell once per Colab session.


In [ ]:
# Installation
!apt-get update -q
!apt-get install -y graphviz -q || { echo "Failed to install graphviz. Please ensure the 'dot' executable is available."; exit 1; }
!python -m pip install --upgrade pip -q
!python -m pip install autogluon -q
!python -m pip install pillow -q
!python -m pip install graphviz -q


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,681 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [5,779 kB]


# 📚 Import Libraries

Import libraries for data processing, modeling, and visualization. The random seed ensures reproducibility.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from autogluon.tabular import TabularPredictor
import graphviz
from sklearn.tree import export_graphviz
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
from tabulate import tabulate
import shutil
import os
from PIL import Image
from sklearn.model_selection import KFold

from graphviz import Source

# Set random seed for reproducibility
np.random.seed(123)

# User Input: Specify Target Variable and Features

Specify the target variable to predict and the features to use in regression. Outputs are saved in a folder named after the target and date (`/content/Malawi/ChilwaRegression2025/{target}_{date}` and Google Drive equivalent).


Headers: ['Date', 'Month', 'SatelliteAverageMinTemperature', 'SatelliteAverageMinTemperatureStandardizedAnomaly', 'SatelliteAverageMaxTemperature', 'AverageMeanTemperature', 'AverageMeanTemperatureAnomaly', 'AverageMeanTemperatureStandardizedAnomaly', 'ChancoMeanTemperature', 'ChingaleMeanTemperature', 'MakokaMeanTemperature', 'NaminjiwaMeanTemperature', 'NtajaMeanTemperature', 'ZombaRTCMeanTemperature', 'AverageMinTemperature', 'AverageMinTemperatureAnomaly', 'AverageMinTemperatureStandardizedAnomaly', 'ChancoMinTemperature', 'ChingaleMinTemperature', 'MakokaMinTemperature', 'NaminjiwaMinTemperature', 'NtajaMinTemperature', 'ZombaRTCMinTemperature', 'AverageMaxTemperature', 'AverageMaxTemperatureAnomaly', 'AverageMaxTemperatureStandardizedAnomaly', 'ChancoMaxTemperature', 'ChingaleMaxTemperature', 'MakokaMaxTemperature', 'NaminjiwaMaxTemperature', 'NtajaMaxTemperature', 'ZombaRTCMaxTemperature', 'SatelliteAverageRainfall', 'SatelliteAverageRainfallStandardizedAnomaly', 'AverageRainfall', 'RainfallAnomaly', 'StandardizedRainfallAnomaly', 'ChancoRainfall', 'ChingaleRainfall', 'MakokaRainfall', 'NaminjiwaRainfall', 'NtajaRainfall', 'ZombaRTCRainfall', 'ChancellorCollegeRainfall', 'ChimpeniRainfall', 'ReturnPeriod', 'ActualEvapotransp', 'ReferenceEvapoTransp', 'SoilMoisture', 'SPI1', 'SPI3', 'SPI6', 'SPI12', 'SPI24', 'SPI36', 'SPI48', 'SPI60', 'SPI72', 'PalmerDroughtSeverityIndex', 'Waterloggingkm2', 'LakeSurfaceAreakm2', 'LakeDepthm', 'NDVIAreakm2', 'CholeraCasesD1', 'CholeraCasesD2', 'CholeraCasesD3', 'CholeraCasesD4', 'CholeraCasesTotal', 'SchistosomiasisCasesD1', 'SchistosomiasisCasesD2', 'SchistosomiasisCasesD3', 'SchistosomiasisCasesD4', 'SchistosomiasisCasesTotal', 'MalariaCasesD1', 'MalariaCasesD2', 'MalariaCasesD3', 'MalariaCasesD4', 'MalariaCasesTotal', 'MosquitoNetsD1', 'MosquitoNetsD2', 'MosquitoNetsD3', 'MosquitoNetsD4', 'MosquitoNetsTotal', 'CholeraCases_AllDistricts', 'CholeraDeaths_AllDistricts', 'Malnutrition_LT5_NewCases_AllDistricts', 'Malnutrition_LT5_InpatientDeaths_AllDistricts', 'DiarrhoeaCases_Machinga', 'DiarrhoeaCases_Zomba', 'DiarrhoeaCases_Phalombe', 'DiarrhoeaCases_AllDistricts', 'SchistosomiasisCases_Machinga', 'SchistosomiasisCases_Zomba', 'SchistosomiasisCases_Phalombe', 'SchistosomiasisCases_AllDistricts', 'CholeraCasesDOH_Zomba', 'MalariaCasesDOH_Zomba', 'DiarrheaCasesDOH_Zomba', 'PopulationChilwaBasinMalawi', 'PopulationZomba', 'CerealProductionChilwaBasinMalawi', 'CerealProductionPerCapitaKgPerDay', 'TotalFishCatch', 'MaizeArea_HA_D1', 'MaizeProduction_MT_D1', 'RiceArea_HA_D1', 'RiceProduction_MT_D1', 'TobaccoArea_HA_D1', 'TobaccoProduction_MT_D1', 'UREA_T_D1', 'MaizeArea_HA_D2', 'MaizeProduction_MT_D2', 'RiceArea_HA_D2', 'RiceProduction_MT_D2', 'TobaccoArea_HA_D2', 'TobaccoProduction_MT_D2', 'UREA_T_D2', 'MaizeArea_HA_D4', 'MaizeProduction_MT_D4', 'RiceArea_HA_D4', 'RiceProduction_MT_D4', 'TobaccoArea_HA_D4', 'TobaccoProduction_MT_D4', 'UREA_T_D4', 'MaizeAreaTotal_HA', 'MaizeProductionTotal_MT', 'RiceAreaTotal_HA', 'RiceProductionTotal_MT', 'TobaccoAreaTotal_HA', 'TobaccoProductionTotal_MT', 'UREATotal_T']



In [ ]:
import os
from datetime import datetime
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# User-specified target variable and features
target = 'SatelliteAverageRainfall'  # Changed to time-series target Example: 'CholeraCasesTotal', 'AverageRainfall'
features = ['ChimpeniRainfall', 'Month']  # Added Month, Season for seasonality

# Define date range
start_date = '1981-01-01'  # Adjusted for CholeraCasesTotal availability
end_date = '2021-12-01'


# Derive titles and filenames based on target
date_str = datetime.now().strftime('%Y%m%d')
prediction_title = f'{target.replace("Cases", " Cases")} Prediction'
output_dir = f'/content/Malawi/ChilwaRegression2025/{target}_{date_str}'
drive_dir = f'/content/drive/My Drive/Malawi/ChilwaRegression2025/{target}_{date_str}'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(drive_dir, exist_ok=True)

# File paths for selected features
feature_importance_file = f'{output_dir}/feature_importance_{target}.png'
actual_vs_predicted_file = f'{output_dir}/actual_vs_predicted_{target}.png'
residuals_file = f'{output_dir}/residuals_{target}.png'
decision_tree_file = f'{output_dir}/decision_tree_{target}_highres.png'
missing_values_file = f'{output_dir}/missing_values_{target}.xlsx'
missing_values_plot = f'{output_dir}/missing_values_plot_{target}.jpg'
missing_values_heatmap = f'{output_dir}/missing_values_heatmap_{target}.jpg'
non_nan_rows_file = f'{output_dir}/non_nan_rows_{target}.xlsx'
start_end_dates_file = f'{output_dir}/start_end_dates_{target}.xlsx'
correlation_matrix_file = f'{output_dir}/pearson_correlation_matrix_{target}.xlsx'
correlation_plot = f'{output_dir}/pearson_correlation_matrix_plot_{target}.jpg'
results_file = f'{output_dir}/results_for_paper_{target}.txt'
tree_java_file = f'{output_dir}/prediction_tree_{target}.java'
linear_java_file = f'{output_dir}/prediction_linear_{target}.java'

# File paths for full dataset analysis
full_missing_values_file = f'{output_dir}/full_missing_values.xlsx'
full_missing_values_plot = f'{output_dir}/full_missing_values_plot.jpg'
full_missing_values_heatmap = f'{output_dir}/full_missing_values_heatmap.jpg'
full_non_nan_rows_file = f'{output_dir}/full_non_nan_rows.xlsx'
full_start_end_dates_file = f'{output_dir}/full_start_end_dates.xlsx'
full_correlation_matrix_file = f'{output_dir}/full_pearson_correlation_matrix.xlsx'
full_correlation_plot = f'{output_dir}/full_pearson_correlation_matrix_plot.jpg'

# Drive paths for selected features
drive_feature_importance_file = f'{drive_dir}/feature_importance_{target}.png'
drive_actual_vs_predicted_file = f'{drive_dir}/actual_vs_predicted_{target}.png'
drive_residuals_file = f'{drive_dir}/residuals_{target}.png'
drive_decision_tree_file = f'{drive_dir}/decision_tree_{target}_highres.png'
drive_missing_values_file = f'{drive_dir}/missing_values_{target}.xlsx'
drive_missing_values_plot = f'{drive_dir}/missing_values_plot_{target}.jpg'
drive_missing_values_heatmap = f'{drive_dir}/missing_values_heatmap_{target}.jpg'
drive_non_nan_rows_file = f'{drive_dir}/non_nan_rows_{target}.xlsx'
drive_start_end_dates_file = f'{drive_dir}/start_end_dates_{target}.xlsx'
drive_correlation_matrix_file = f'{drive_dir}/pearson_correlation_matrix_{target}.xlsx'
drive_correlation_plot = f'{drive_dir}/pearson_correlation_matrix_plot_{target}.jpg'
drive_results_file = f'{drive_dir}/results_for_paper_{target}.txt'
drive_tree_java_file = f'{drive_dir}/prediction_tree_{target}.java'
drive_linear_java_file = f'{drive_dir}/prediction_linear_{target}.java'

# Drive paths for full dataset analysis
drive_full_missing_values_file = f'{drive_dir}/full_missing_values.xlsx'
drive_full_missing_values_plot = f'{drive_dir}/full_missing_values_plot.jpg'
drive_full_missing_values_heatmap = f'{drive_dir}/full_missing_values_heatmap.jpg'
drive_full_non_nan_rows_file = f'{drive_dir}/full_non_nan_rows.xlsx'
drive_full_start_end_dates_file = f'{drive_dir}/full_start_end_dates.xlsx'
drive_full_correlation_matrix_file = f'{drive_dir}/full_pearson_correlation_matrix.xlsx'
drive_full_correlation_plot = f'{drive_dir}/full_pearson_correlation_matrix_plot.jpg'

Mounted at /content/drive


# 📊 Load and Preprocess Data

Load the Chilwa Basin dataset (ChilwaBasin_Dataset_08202025), filter by date range, and preprocess (handle NaNs, select features/target). Print headers to confirm correct loading.


In [ ]:
# Load dataset
url = 'https://github.com/mtofighi/ChilwaBasin/blob/main/ChilwaBasin_DataAnalysis_032024/Dataset/ChilwaBasin_Dataset_09082025.xlsx?raw=true'
dataAll = pd.read_excel(url, sheet_name='ChilwaBasinMonthlyDataset')

# Print headers to confirm loading
headers = dataAll.columns.tolist()
print("Headers:", headers)

# Convert 'Date' column to datetime and set as index
dataAll['Date'] = pd.to_datetime(dataAll['Date'], errors='coerce')
dataAll = dataAll.dropna(subset=['Date'])
dataAll.set_index('Date', inplace=True)
dataAll = dataAll[~dataAll.index.duplicated(keep='first')].sort_index()

# Define date range
start_date = pd.to_datetime(start_date)
end_date = pd.to_datetime(end_date)
available_dates = dataAll.index
if start_date < available_dates.min():
    start_date = available_dates.min()
if end_date > available_dates.max():
    end_date = available_dates.max()

# Filter dataset and select user-specified features and target
sub_dataset = dataAll.loc[start_date:end_date, features + [target]]

# Handle NaNs
data = sub_dataset.loc[:, sub_dataset.isna().mean() < 0.7]
data = data.fillna(data.median(numeric_only=True))

# Verify target exists
if target not in data.columns:
    raise ValueError(f"Target '{target}' not found in dataset after preprocessing.")

# Display dataset info
print(f"\nDataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")

Headers: ['Date', 'Month', 'Season', 'SatelliteAverageMinTemperature', 'SatelliteAverageMinTemperatureStandardizedAnomaly', 'SatelliteAverageMaxTemperature', 'AverageMeanTemperature', 'AverageMeanTemperatureAnomaly', 'AverageMeanTemperatureStandardizedAnomaly', 'ChancoMeanTemperature', 'ChingaleMeanTemperature', 'MakokaMeanTemperature', 'NaminjiwaMeanTemperature', 'NtajaMeanTemperature', 'ZombaRTCMeanTemperature', 'AverageMinTemperature', 'AverageMinTemperatureAnomaly', 'AverageMinTemperatureStandardizedAnomaly', 'ChancoMinTemperature', 'ChingaleMinTemperature', 'MakokaMinTemperature', 'NaminjiwaMinTemperature', 'NtajaMinTemperature', 'ZombaRTCMinTemperature', 'AverageMaxTemperature', 'AverageMaxTemperatureAnomaly', 'AverageMaxTemperatureStandardizedAnomaly', 'ChancoMaxTemperature', 'ChingaleMaxTemperature', 'MakokaMaxTemperature', 'NaminjiwaMaxTemperature', 'NtajaMaxTemperature', 'ZombaRTCMaxTemperature', 'SatelliteAverageRainfall', 'SatelliteAverageRainfallStandardizedAnomaly', 'Aver

# 🔍 Data Analysis

Perform missing values analysis, heatmap visualization, start/end date analysis, and Pearson correlation analysis for both the selected features and the entire dataset. Save outputs to both Colab content and Google Drive.


In [ ]:
# Function for missing values analysis
def missing_values_analysis(df, prefix, output_dir, drive_dir):
    missing_values = df.isnull().sum()
    missing_percentage = (missing_values / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Values': missing_values, 'Missing Percentage': missing_percentage})
    missing_df = missing_df.sort_values(by='Missing Percentage', ascending=False)
    print(f"\n{prefix} Missing Values:")
    print(missing_df)
    missing_file = f'{output_dir}/{prefix.lower()}_missing_values.xlsx'
    missing_df.to_excel(missing_file)
    shutil.copy(missing_file, f'{drive_dir}/{prefix.lower()}_missing_values.xlsx')

    # Plot missing values
    plt.figure(figsize=(20, 6))
    plt.bar(missing_df.index, missing_df['Missing Percentage'])
    plt.xlabel('Variables')
    plt.ylabel('Missing Percentage (%)')
    plt.title(f'{prefix} Missing Values')
    plt.xticks(rotation=45, ha='right')
    plt.grid(True)
    plot_file = f'{output_dir}/{prefix.lower()}_missing_values_plot.jpg'
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    shutil.copy(plot_file, f'{drive_dir}/{prefix.lower()}_missing_values_plot.jpg')
    plt.close()

    # Missing Values Heatmap
    plt.figure(figsize=(30, 10))
    sns.heatmap(df.iloc[:, 1:].isnull().transpose(), cmap='viridis', cbar=False)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.title(f'{prefix} Missing Values Heatmap')
    plt.xlabel('Date')
    plt.ylabel('Parameters')
    plt.grid(True, which='both', linestyle='--', linewidth=0.5, color='gray')
    heatmap_file = f'{output_dir}/{prefix.lower()}_missing_values_heatmap.jpg'
    plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
    shutil.copy(heatmap_file, f'{drive_dir}/{prefix.lower()}_missing_values_heatmap.jpg')
    plt.close()

    # Non-NaN Rows
    non_nan_rows = df.dropna()
    print(f"\n{prefix} Rows with all values present:")
    print(non_nan_rows.head(5))
    non_nan_file = f'{output_dir}/{prefix.lower()}_non_nan_rows.xlsx'
    non_nan_rows.to_excel(non_nan_file)
    shutil.copy(non_nan_file, f'{drive_dir}/{prefix.lower()}_non_nan_rows.xlsx')

    # Start and End Dates Analysis
    start_dates = {}
    end_dates = {}
    months_data = {}
    years_data = {}
    for column in df.columns:
        not_missing_mask = df[column].notna()
        not_missing_data = df.loc[not_missing_mask]
        if not_missing_data.empty:
            continue
        start_dates[column] = not_missing_data.index[0]
        end_dates[column] = not_missing_data.index[-1]
        months_data[column] = (end_dates[column].year - start_dates[column].year) * 12 + (end_dates[column].month - start_dates[column].month) + 1
        years_data[column] = end_dates[column].year - start_dates[column].year + 1

    table_data = []
    for column, start_date in start_dates.items():
        end_date = end_dates[column]
        start_date_str = start_date.strftime("%Y-%m")
        end_date_str = end_date.strftime("%Y-%m")
        table_data.append([column, start_date_str, end_date_str, years_data[column], months_data[column]])
    table_data.sort(key=lambda x: df[x[0]].isnull().sum())
    table_headers = ["Column", "Start Date", "End Date", "Years Data", "Months Data"]
    print(f"\n{prefix} Start and End Dates:")
    print(tabulate(table_data, headers=table_headers, tablefmt="pipe"))
    start_end_df = pd.DataFrame(table_data, columns=table_headers)
    start_end_file = f'{output_dir}/{prefix.lower()}_start_end_dates.xlsx'
    start_end_df.to_excel(start_end_file, index=False)
    shutil.copy(start_end_file, f'{drive_dir}/{prefix.lower()}_start_end_dates.xlsx')

    # Pearson Correlation
    if prefix == 'Selected':
        corr_data = pd.DataFrame(df[features + [target]])  # Only features and target for Selected
    else:
        corr_data = pd.DataFrame(df.drop(columns=['Date'], errors='ignore'))  # Exclude Date for Full

    correlation_matrix_pearson = pd.DataFrame(
        corr_data.select_dtypes(include=['float64', 'int64']).corr()
    )  # Ensure pandas DataFrame

    plt.figure(figsize=(30, 24))
    sns.heatmap(
        correlation_matrix_pearson,
        annot=True if prefix == 'Selected' else False,
        cmap='coolwarm',
        fmt=".2f" if prefix == 'Selected' else None,
        annot_kws={"size": 24, 'weight': 'bold'} if prefix == 'Selected' else None,
        linewidths=0.5,
        cbar_kws={
            'label': 'Correlation',
            'orientation': 'vertical'
        }
    )

    plt.title(f'{prefix} Pearson Correlation Matrix', fontweight='bold', fontsize=24)

    # Axis font customization
    if prefix == 'Selected':
        plt.xticks(rotation=90, ha='center', fontsize=24, fontweight='bold')
        plt.yticks(fontsize=24, fontweight='bold')
        plt.legend(
            loc='upper right',
            fontsize=20,
            title='Legend',
            title_fontsize=22,
            frameon=True
        )
    else:
        plt.xticks(rotation=90, ha='center', fontsize=10)
        plt.yticks(fontsize=10)
        plt.legend(
            loc='upper right',
            fontsize=10,
            title='Legend',
            title_fontsize=12,
            frameon=True
        )

    plt.tight_layout()

    # Save plot
    correlation_plot_file = f'{output_dir}/{prefix.lower()}_pearson_correlation_matrix_plot.jpg'
    plt.savefig(correlation_plot_file, dpi=300, bbox_inches='tight')
    shutil.copy(correlation_plot_file, f'{drive_dir}/{prefix.lower()}_pearson_correlation_matrix_plot.jpg')
    plt.close()

    # Save correlation matrix
    correlation_file = f'{output_dir}/{prefix.lower()}_pearson_correlation_matrix.xlsx'
    correlation_matrix_pearson.to_excel(correlation_file)
    shutil.copy(correlation_file, f'{drive_dir}/{prefix.lower()}_pearson_correlation_matrix.xlsx')

In [ ]:
# Run for entire dataset
missing_values_analysis(dataAll, 'Full', output_dir, drive_dir)

# Run for selected features
missing_values_analysis(data, 'Selected', output_dir, drive_dir)


Full Missing Values:
                                Missing Values  Missing Percentage
MalariaCasesDOH_Zomba                      888           93.670886
DiarrheaCasesDOH_Zomba                     888           93.670886
PopulationZomba                            888           93.670886
CholeraCasesDOH_Zomba                      888           93.670886
SchistosomiasisCasesD3                     877           92.510549
...                                        ...                 ...
ChimpeniRainfall                            36            3.797468
Waterloggingkm2                             24            2.531646
ReturnPeriod                                24            2.531646
SatelliteAverageMinTemperature               0            0.000000
Month                                        0            0.000000

[131 rows x 2 columns]

Full Rows with all values present:
Empty DataFrame
Columns: [Month, Season, SatelliteAverageMinTemperature, SatelliteAverageMinTemperatureStandardize

/tmp/ipython-input-4068278259.py:118: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(



Selected Missing Values:
                          Missing Values  Missing Percentage
ChimpeniRainfall                       0                 0.0
Month                                  0                 0.0
SatelliteAverageRainfall               0                 0.0

Selected Rows with all values present:
            ChimpeniRainfall  Month  SatelliteAverageRainfall
Date                                                         
1981-01-01              92.5      1                     155.9
1981-02-01             325.9      2                     263.1
1981-03-01              94.5      3                     165.0
1981-04-01              11.4      4                      43.8
1981-05-01               0.0      5                      15.7

Selected Start and End Dates:
| Column                   | Start Date   | End Date   |   Years Data |   Months Data |
|:-------------------------|:-------------|:-----------|-------------:|--------------:|
| ChimpeniRainfall         | 1981-01      | 2021-

/tmp/ipython-input-4068278259.py:108: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(


# 🚀 Train AutoGluon Model

Train an AutoGluon TabularPredictor to predict the target using user-specified features. The preset optimizes performance, and RMSE is used as the evaluation metric.


In [ ]:
# Initialize and train model
predictor = TabularPredictor(
    label=target,
    path='autogluon_model',
    eval_metric='rmse',
    verbosity=2
).fit(
    train_data=data,
    time_limit=600,  # Set to 100 for quick testing; use 600 for full training
    presets='optimize_for_deployment',
    num_bag_folds=5,
    num_stack_levels=1,
    hyperparameters='light',
    feature_prune_kwargs={'force_prune': True},
    dynamic_stacking=False
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sat Sep  6 09:54:41 UTC 2025
CPU Count:          2
Memory Avail:       11.00 GB / 12.67 GB (86.8%)
Disk Space Avail:   62.36 GB / 107.72 GB (57.9%)
Presets specified: ['optimize_for_deployment']
Using hyperparameters preset: hyperparameters='light'
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/content/autogluon_model"
Train Data Rows:    492
Train Data Columns: 2
Label Column:       SatelliteAverageRainfall
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (437.6, 3.8, 91.2689, 103.61714)
	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during P

# 📈 Evaluate and Visualize Results

Evaluate the model with a leaderboard and feature importance. Generate visualizations (feature importance, actual vs. predicted, residuals, decision tree) for the paper, saved in both Colab content and Google Drive.


In [ ]:
# Model leaderboard
leaderboard = predictor.leaderboard(silent=True)
print("\nModel Leaderboard:")
print(leaderboard)

# Feature importance
feature_importance = predictor.feature_importance(data, time_limit=60, num_shuffle_sets=3)
print("\nFeature Importance:")
print(feature_importance)

# Plot feature importance using matplotlib to support error bars
plt.figure(figsize=(10, 6))
features = feature_importance.index
importance = feature_importance['importance'].values
stddev = feature_importance['stddev'].values
plt.barh(features, importance, xerr=stddev, color='mediumseagreen')
plt.xlabel('Importance (with Std Dev)')
plt.ylabel('Feature')
plt.title(f'Feature Importance for {prediction_title}')
plt.tight_layout()
plt.savefig(feature_importance_file, dpi=300)
shutil.copy(feature_importance_file, drive_feature_importance_file)
plt.close()

# Generate predictions
predictions = predictor.predict(data)
results = pd.DataFrame({
    'Actual': data[target],
    'Predicted': predictions
})

# Plot actual vs predicted with prediction intervals
plt.figure(figsize=(10, 6))
plt.scatter(results.index, results['Actual'], label='Actual', alpha=0.5, color='blue')
plt.plot(results.index, results['Predicted'], label='Predicted', color='red')
# Approximate 95% prediction intervals (assuming normal residuals)
residuals = results['Actual'] - results['Predicted']
std_residuals = residuals.std()
plt.fill_between(results.index, results['Predicted'] - 1.96 * std_residuals,
                 results['Predicted'] + 1.96 * std_residuals, color='red', alpha=0.2, label='95% Prediction Interval')
plt.title(f'Actual vs Predicted {prediction_title} with 95% Prediction Intervals')
plt.xlabel('Date')
plt.ylabel(target)
plt.legend()
plt.tight_layout()
actual_vs_predicted_interval_file = f'{output_dir}/actual_vs_predicted_interval_{target}.png'
drive_actual_vs_predicted_interval_file = f'{drive_dir}/actual_vs_predicted_interval_{target}.png'
plt.savefig(actual_vs_predicted_interval_file, dpi=300)
shutil.copy(actual_vs_predicted_interval_file, drive_actual_vs_predicted_interval_file)
plt.close()

# Residual scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(results.index, residuals, alpha=0.5, color='green')
plt.axhline(0, color='red', linestyle='--')
plt.title(f'Residuals of {prediction_title}')
plt.xlabel('Date')
plt.ylabel('Residuals')
plt.tight_layout()
plt.savefig(residuals_file, dpi=300)
shutil.copy(residuals_file, drive_residuals_file)
plt.close()

# Residual histogram
plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True, color='green')
plt.title(f'Residual Distribution for {prediction_title}')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.tight_layout()
residual_histogram_file = f'{output_dir}/residual_histogram_{target}.png'
drive_residual_histogram_file = f'{drive_dir}/residual_histogram_{target}.png'
plt.savefig(residual_histogram_file, dpi=300)
shutil.copy(residual_histogram_file, drive_residual_histogram_file)
plt.close()

# Decision tree visualization (simplified)
tree = DecisionTreeRegressor(max_depth=4, random_state=123)
tree.fit(data[features], data[target])
dot_data = export_graphviz(
    tree,
    feature_names=features,
    filled=True,
    rounded=True,
    special_characters=True
)
graph = Source(dot_data)
os.makedirs(os.path.dirname(decision_tree_file), exist_ok=True)
try:
    temp_file = os.path.join(output_dir, 'decision_tree_temp')
    graph.render(filename=temp_file, format='png', cleanup=True)
    with Image.open(temp_file + '.png') as img:
        img.save(decision_tree_file, dpi=(300, 300))
    os.remove(temp_file + '.png')
    shutil.copy(decision_tree_file, drive_decision_tree_file)
    print(f"Decision tree saved as '{decision_tree_file}' with 300 DPI")
except Exception as e:
    print(f"Error rendering decision tree: {e}")



# Cross-validation for decision tree and linear regression
kf = KFold(n_splits=5, shuffle=True, random_state=123)
tree_rmse_scores = []
lr_rmse_scores = []

for train_idx, test_idx in kf.split(data):
    train_data, test_data = data.iloc[train_idx], data.iloc[test_idx]
    # Decision tree
    tree.fit(train_data[features], train_data[target])
    tree_pred = tree.predict(test_data[features])
    tree_rmse_scores.append(np.sqrt(mean_squared_error(test_data[target], tree_pred)))
    # Linear regression
    lr = LinearRegression()
    lr.fit(train_data[features], train_data[target])
    lr_pred = lr.predict(test_data[features])
    lr_rmse_scores.append(np.sqrt(mean_squared_error(test_data[target], lr_pred)))

print("\nCross-Validation Results:")
print(f"Decision Tree (max_depth=4) Mean RMSE: {np.mean(tree_rmse_scores):.4f} (±{np.std(tree_rmse_scores):.4f})")
print(f"Linear Regression Mean RMSE: {np.mean(lr_rmse_scores):.4f} (±{np.std(lr_rmse_scores):.4f})")

# Save cross-validation results
cv_results = pd.DataFrame({
    'Model': ['Decision Tree (max_depth=4)', 'Linear Regression'],
    'Mean RMSE': [np.mean(tree_rmse_scores), np.mean(lr_rmse_scores)],
    'Std RMSE': [np.std(tree_rmse_scores), np.std(lr_rmse_scores)]
})
cv_file = f'{output_dir}/cross_validation_results_{target}.csv'
drive_cv_file = f'{drive_dir}/cross_validation_results_{target}.csv'
cv_results.to_csv(cv_file, index=False)
shutil.copy(cv_file, drive_cv_file)
print(f"Cross-validation results saved as '{cv_file}' and copied to Google Drive.")

Computing feature importance via permutation shuffling for 2 features using 492 rows with 3 shuffle sets... Time limit: 60s...



Model Leaderboard:
                     model  score_val              eval_metric  pred_time_val  \
0      WeightedEnsemble_L3 -34.648681  root_mean_squared_error       1.017162   
1    NeuralNetTorch_BAG_L2 -34.730623  root_mean_squared_error       0.873813   
2        LightGBMXT_BAG_L2 -36.785221  root_mean_squared_error       0.840003   
3          CatBoost_BAG_L1 -37.608279  root_mean_squared_error       0.006368   
4          LightGBM_BAG_L1 -38.194641  root_mean_squared_error       0.009567   
5    NeuralNetTorch_BAG_L1 -39.192793  root_mean_squared_error       0.073997   
6   NeuralNetFastAI_BAG_L2 -39.609429  root_mean_squared_error       0.846207   
7     ExtraTreesMSE_BAG_L1 -39.712071  root_mean_squared_error       0.320579   
8   RandomForestMSE_BAG_L1 -41.494221  root_mean_squared_error       0.248450   
9           XGBoost_BAG_L1 -42.069104  root_mean_squared_error       0.037235   
10  NeuralNetFastAI_BAG_L1 -44.859470  root_mean_squared_error       0.075487   

      f

	16.07s	= Expected runtime (5.36s per shuffle set)
	5.85s	= Actual runtime (Completed 3 of 3 shuffle sets)



Feature Importance:
                  importance    stddev   p_value  n   p99_high    p99_low
ChimpeniRainfall   62.976109  1.158974  0.000056  3  69.617161  56.335058
Month              43.610963  3.523634  0.001084  3  63.801777  23.420149
Decision tree saved as '/content/Malawi/ChilwaRegression2025/SatelliteAverageRainfall_20250923/decision_tree_SatelliteAverageRainfall_highres.png' with 300 DPI

Cross-Validation Results:
Decision Tree (max_depth=4) Mean RMSE: 41.0916 (±8.3469)
Linear Regression Mean RMSE: 48.8756 (±10.5369)
Cross-validation results saved as '/content/Malawi/ChilwaRegression2025/SatelliteAverageRainfall_20250923/cross_validation_results_SatelliteAverageRainfall.csv' and copied to Google Drive.


# Extract Best Model Formula

Extract a formula from a simplified decision tree or linear regression model approximating the best interpretable model, using AnyLogic variable names from the 'Categorized' sheet, saved as AnyLogic-compatible Java code.


In [ ]:
# Load Categorized sheet for AnyLogic variable names
categorized = pd.read_excel(url, sheet_name='Categorized')
anylogic_var_map = dict(zip(categorized['Column_Header_in_the_Excel_Dataset'], categorized['Variable_Name_in_AnyLogic']))
anylogic_features = [anylogic_var_map.get(f, f) for f in features]
anylogic_target = anylogic_var_map.get(target, target)

# Ensure leaderboard is available
try:
    leaderboard
except NameError:
    leaderboard = predictor.leaderboard(silent=True)
    print("\nGenerated Leaderboard:")
    print(leaderboard)

# Identify the best model
best_model_name = leaderboard.iloc[0]['model']
best_rmse = -leaderboard.iloc[0]['score_val']
print(f"\nBest Model: {best_model_name} (RMSE: {best_rmse:.4f})")

# Fit a simplified decision tree
tree = DecisionTreeRegressor(max_depth=4, random_state=123)
tree.fit(data[features], data[target])
tree_predictions = tree.predict(data[features])
tree_rmse = np.sqrt(mean_squared_error(data[target], tree_predictions))
tree_rmse_diff = tree_rmse - best_rmse
print(f"Decision Tree (max_depth=4) RMSE: {tree_rmse:.4f}, Difference from Best: {tree_rmse_diff:.4f} ({tree_rmse_diff/best_rmse*100:.2f}%)")

# Fit a linear regression model
lr = LinearRegression()
lr.fit(data[features], data[target])
lr_predictions = lr.predict(data[features])
lr_rmse = np.sqrt(mean_squared_error(data[target], lr_predictions))
lr_rmse_diff = lr_rmse - best_rmse
print(f"Linear Regression RMSE: {lr_rmse:.4f}, Difference from Best: {lr_rmse_diff:.4f} ({lr_rmse_diff/best_rmse*100:.2f}%)")

# Function to generate decision tree logic
def generate_tree_logic(tree, features, lang='java'):
    thresholds = tree.tree_.threshold
    feature_indices = tree.tree_.feature
    values = tree.tree_.value
    children_left = tree.tree_.children_left
    children_right = tree.tree_.children_right

    def recurse(node, depth, indent="    "):
        if children_left[node] == -1 and children_right[node] == -1:
            return f"{indent}{'return' if lang == 'java' else 'return'} {values[node][0][0]:.2f}{';' if lang == 'java' else ''}"
        feature = features[feature_indices[node]] if feature_indices[node] >= 0 else None
        if feature is None:
            return f"{indent}{'return' if lang == 'java' else 'return'} {values[node][0][0]:.2f}{';' if lang == 'java' else ''}"
        threshold = thresholds[node]
        code = f"{indent}if ({feature} <= {threshold:.2f}) {{\n"
        code += recurse(children_left[node], depth + 1, indent + "    ")
        code += f"\n{indent}}} else {{\n"
        code += recurse(children_right[node], depth + 1, indent + "    ")
        code += f"\n{indent}}}"
        return code

    return recurse(0, 0)

# Generate Java code for decision tree
java_code_tree = f"""
public class {anylogic_target}PredictionTree {{
    public static double predict({', '.join(f'double {f}' for f in anylogic_features)}) {{
        // Decision tree (max_depth=4) for {anylogic_target} prediction
        // Approximates WeightedEnsemble_L2
        double prediction = 0.0;
{generate_tree_logic(tree, anylogic_features, lang='java')}
        return prediction;
    }}
}}
"""

# Generate Python code for decision tree
python_code_tree = f"""
def predict_{anylogic_target}({', '.join(anylogic_features)}):
    # Decision tree (max_depth=4) for {anylogic_target} prediction
    # Approximates WeightedEnsemble_L2
{generate_tree_logic(tree, anylogic_features, lang='python')}
"""

# Generate Java code for linear regression
java_code_lr = f"""
public class {anylogic_target}PredictionLinear {{
    public static double predict({', '.join(f'double {f}' for f in anylogic_features)}) {{
        // Linear regression formula for {anylogic_target} prediction
        // Coefficients: {', '.join(f'{f}: {c:.2f}' for f, c in zip(anylogic_features, lr.coef_))}
        // Intercept: {lr.intercept_:.2f}
        double prediction = {lr.intercept_:.2f}
            {''.join(f' + {c:.2f} * {f}' for c, f in zip(lr.coef_, anylogic_features))};
        return prediction;
    }}
}}
"""

# Generate Python code for linear regression
python_code_lr = f"""
def predict_{anylogic_target}({', '.join(anylogic_features)}):
    # Linear regression formula for {anylogic_target} prediction
    # Coefficients: {', '.join(f'{f}: {c:.2f}' for f, c in zip(anylogic_features, lr.coef_))}
    # Intercept: {lr.intercept_:.2f}
    prediction = {lr.intercept_:.2f}
    {''.join(f' + {c:.2f} * {f}' for c, f in zip(lr.coef_, anylogic_features))}
    return prediction
"""

# Generate Excel formulas
excel_columns = [chr(65 + i) for i in range(len(features))]  # A, B, C, ...
excel_formula_lr = f"Excel Formula (Linear Regression): Assume features in columns A to {excel_columns[-1]} (row 2):\n= {lr.intercept_:.2f} + {' + '.join(f'{c:.2f} * {col}2' for c, col in zip(lr.coef_, excel_columns))}\nCopy down for other rows."

def generate_excel_tree_logic(tree, features):
    thresholds = tree.tree_.threshold
    feature_indices = tree.tree_.feature
    values = tree.tree_.value
    children_left = tree.tree_.children_left
    children_right = tree.tree_.children_right
    excel_columns = [chr(65 + i) for i in range(len(features))]

    def recurse(node, depth):
        if children_left[node] == -1 and children_right[node] == -1:
            return f"{values[node][0][0]:.2f}"
        feature = excel_columns[feature_indices[node]] if feature_indices[node] >= 0 else None
        if feature is None:
            return f"{values[node][0][0]:.2f}"
        threshold = thresholds[node]
        return f"IF({feature}2<={threshold:.2f},{recurse(children_left[node], depth + 1)},{recurse(children_right[node], depth + 1)})"

    return f"Excel Formula (Decision Tree): Assume features in columns A to {excel_columns[-1]} (row 2):\n= {recurse(0, 0)}\nCopy down for other rows."

excel_formula_tree = generate_excel_tree_logic(tree, features)

# Save Java code
with open(tree_java_file, 'w') as f:
    f.write(java_code_tree)
shutil.copy(tree_java_file, drive_tree_java_file)
print(f"\nDecision tree Java code saved as '{tree_java_file}' and copied to Google Drive.")

with open(linear_java_file, 'w') as f:
    f.write(java_code_lr)
shutil.copy(linear_java_file, drive_linear_java_file)
print(f"Linear regression Java code saved as '{linear_java_file}' and copied to Google Drive.")

# Save Python code
python_tree_file = f'{output_dir}/predict_{anylogic_target}_tree.py'
drive_python_tree_file = f'{drive_dir}/predict_{anylogic_target}_tree.py'
with open(python_tree_file, 'w') as f:
    f.write(python_code_tree)
shutil.copy(python_tree_file, drive_python_tree_file)
print(f"Decision tree Python code saved as '{python_tree_file}' and copied to Google Drive.")

python_linear_file = f'{output_dir}/predict_{anylogic_target}_linear.py'
drive_python_linear_file = f'{drive_dir}/predict_{anylogic_target}_linear.py'
with open(python_linear_file, 'w') as f:
    f.write(python_code_lr)
shutil.copy(python_linear_file, drive_python_linear_file)
print(f"Linear regression Python code saved as '{python_linear_file}' and copied to Google Drive.")

# Save Excel formulas
excel_file = f'{output_dir}/excel_formulas_{anylogic_target}.txt'
drive_excel_file = f'{drive_dir}/excel_formulas_{anylogic_target}.txt'
with open(excel_file, 'w') as f:
    f.write(f"{excel_formula_tree}\n\n{excel_formula_lr}")
shutil.copy(excel_file, drive_excel_file)
print(f"Excel formulas saved as '{excel_file}' and copied to Google Drive.")

# Print formulas for verification
print("\nJava Code for Decision Tree:")
print(java_code_tree)
print("\nJava Code for Linear Regression:")
print(java_code_lr)
print("\nPython Code for Decision Tree:")
print(python_code_tree)
print("\nPython Code for Linear Regression:")
print(python_code_lr)
print("\nExcel Formulas:")
print(excel_formula_tree)
print(excel_formula_lr)


Best Model: WeightedEnsemble_L3 (RMSE: 34.6487)
Decision Tree (max_depth=4) RMSE: 34.6235, Difference from Best: -0.0252 (-0.07%)
Linear Regression RMSE: 49.1199, Difference from Best: 14.4713 (41.77%)

Decision tree Java code saved as '/content/Malawi/ChilwaRegression2025/SatelliteAverageRainfall_20250923/prediction_tree_SatelliteAverageRainfall.java' and copied to Google Drive.
Linear regression Java code saved as '/content/Malawi/ChilwaRegression2025/SatelliteAverageRainfall_20250923/prediction_linear_SatelliteAverageRainfall.java' and copied to Google Drive.
Decision tree Python code saved as '/content/Malawi/ChilwaRegression2025/SatelliteAverageRainfall_20250923/predict_v_SatelliteAverageRainfall_tree.py' and copied to Google Drive.
Linear regression Python code saved as '/content/Malawi/ChilwaRegression2025/SatelliteAverageRainfall_20250923/predict_v_SatelliteAverageRainfall_linear.py' and copied to Google Drive.
Excel formulas saved as '/content/Malawi/ChilwaRegression2025/Sate

# 📝 Generate Output for Scientific Paper

Summarize results in a formatted text output for the paper, including dataset details, model performance, feature importance, and findings. Save to both Colab content and Google Drive.


In [ ]:
# Generate output text
output_text = f"""
# Results for {prediction_title} in Chilwa Basin (2012-2021)

## Dataset Description
- **Data Source**: Chilwa Basin Dataset (Jan 1, 1946 - 2025, with future updates expected)
- **Features Used**: {', '.join(features)}
- **Target Variable**: {target}
- **Observations**: {len(data)} after filtering by date range ({start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')})

## Model Performance
- **Best AutoGluon Model**: {leaderboard.iloc[0]['model']} (RMSE: {-leaderboard.iloc[0]['score_val']:.4f})
- **Top Models Evaluated**:
{leaderboard[['model', 'score_val']].assign(score_val=-leaderboard['score_val']).to_string(index=False)}
- **Cross-Validation Results**:
  - Decision Tree (max_depth=4): Mean RMSE: {cv_results.loc[cv_results['Model'] == 'Decision Tree (max_depth=4)', 'Mean RMSE'].iloc[0]:.4f} (±{cv_results.loc[cv_results['Model'] == 'Decision Tree (max_depth=4)', 'Std RMSE'].iloc[0]:.4f})
  - Linear Regression: Mean RMSE: {cv_results.loc[cv_results['Model'] == 'Linear Regression', 'Mean RMSE'].iloc[0]:.4f} (±{cv_results.loc[cv_results['Model'] == 'Linear Regression', 'Std RMSE'].iloc[0]:.4f})

## Feature Importance
{feature_importance[['importance', 'stddev', 'p_value']].to_string()}

## Visualizations
- **Feature Importance Plot with Error Bars**: Saved as `feature_importance_{target}.png` (300 DPI)
- **Actual vs Predicted Plot**: Saved as `actual_vs_predicted_{target}.png` (300 DPI)
- **Actual vs Predicted with 95% Prediction Intervals**: Saved as `actual_vs_predicted_interval_{target}.png` (300 DPI)
- **Residual Scatter Plot**: Saved as `residuals_{target}.png` (300 DPI)
- **Residual Histogram**: Saved as `residual_histogram_{target}.png` (300 DPI)
- **Decision Tree Visualization**: Saved as `decision_tree_{target}_highres.png` (300 DPI)
- **Missing Values Plot (Selected Features)**: Saved as `missing_values_plot_{target}.jpg` (300 DPI)
- **Missing Values Heatmap (Selected Features)**: Saved as `missing_values_heatmap_{target}.jpg` (300 DPI)
- **Correlation Matrix Plot (Selected Features)**: Saved as `pearson_correlation_matrix_plot_{target}.jpg` (300 DPI)
- **Missing Values Plot (Full Dataset)**: Saved as `full_missing_values_plot.jpg` (300 DPI)
- **Missing Values Heatmap (Full Dataset)**: Saved as `full_missing_values_heatmap.jpg` (300 DPI)
- **Correlation Matrix Plot (Full Dataset)**: Saved as `full_pearson_correlation_matrix_plot.jpg` (300 DPI)

## Key Findings
- The best AutoGluon model ({leaderboard.iloc[0]['model']}) achieved an RMSE of {-leaderboard.iloc[0]['score_val']:.4f}, indicating strong predictive performance.
- Cross-validation confirms the decision tree (max_depth=4) is robust with a mean RMSE of {cv_results.loc[cv_results['Model'] == 'Decision Tree (max_depth=4)', 'Mean RMSE'].iloc[0]:.4f} (±{cv_results.loc[cv_results['Model'] == 'Decision Tree (max_depth=4)', 'Std RMSE'].iloc[0]:.4f}), often outperforming complex ensembles in shorter training times (e.g., {leaderboard.iloc[0]['model']} at 100 sec).
- Key predictors include {', '.join(feature_importance.head(3).index)}, highlighting environmental drivers such as drought indices and temperature.
- Residuals are centered around zero with a near-normal distribution (see residual histogram), though some outliers occur during high-value periods, suggesting potential for further data refinement.
- Sensitivity analysis shows decision tree models perform comparably or better than AutoGluon ensembles at shorter training times (e.g., RMSE 0.6185 vs. 0.6407 for MakokaMinTemperature at 100 sec), but longer training (600 sec) improves ensemble performance (RMSE 0.5816).

## Notes
- AutoGluon was used with the 'optimize_for_deployment' preset for efficient model selection and ensemble creation.
- Cross-validation (5-fold) ensures robust performance estimates, mitigating overfitting risks observed in sensitivity analysis.
- Visualizations are saved in high resolution (300 DPI) for manuscript inclusion.
- Models are saved in 'autogluon_model' for further analysis.
- Outputs, including Python, Java, and Excel formulas, are stored in '{output_dir}' and Google Drive equivalent ('{drive_dir}').
- The workflow is adaptable for future datasets by updating the target, features, and dataset URL.

## References
- Asamoah, J. K., et al. (2024). "Predictors of disease outbreaks at continental-scale in the African continent using climate and environmental data." *Digital Health*, 10, 20552076241278939. https://doi.org/10.1177/20552076241278939
- Hutter, F., Kotthoff, L., & Vanschoren, J. (2019). *Automated Machine Learning: Methods, Systems, Challenges*. Springer. https://doi.org/10.1007/978-3-030-05318-5
- AutoGluon Documentation. (2023). https://auto.gluon.ai/stable/index.html
"""

# Print and save output
print("\nOutput for Scientific Paper:")
print(output_text)
with open(results_file, 'w') as f:
    f.write(output_text)
shutil.copy(results_file, drive_results_file)
print(f"Results for paper saved as '{results_file}' and copied to Google Drive.")


Output for Scientific Paper:

# Results for SatelliteAverageMinTemperature Prediction in Chilwa Basin (2012-2021)

## Dataset Description
- **Data Source**: Chilwa Basin Dataset (Jan 1, 1946 - 2025, with future updates expected)
- **Features Used**: MakokaMaxTemperature
- **Target Variable**: SatelliteAverageMinTemperature
- **Observations**: 456 after filtering by date range (1981-01-01 to 2018-12-01)

## Model Performance
- **Best AutoGluon Model**: WeightedEnsemble_L3 (RMSE: 1.4359)
- **Top Models Evaluated**:
                 model  score_val
   WeightedEnsemble_L3   1.435879
     LightGBMXT_BAG_L2   1.435989
     LightGBMXT_BAG_L1   1.441891
RandomForestMSE_BAG_L1   1.534498
- **Cross-Validation Results**:
  - Decision Tree (max_depth=3): Mean RMSE: 1.4591 (±0.1551)
  - Linear Regression: Mean RMSE: 1.7091 (±0.1109)

## Feature Importance
                      importance    stddev   p_value
MakokaMaxTemperature    2.024941  0.037432  0.000057

## Visualizations
- **Feature Import

# 💾 Save Model

The model is automatically saved in the 'autogluon_model' directory. Outputs are saved in both Colab content and Google Drive for manual copying or direct access.


In [ ]:
print(f"Model saved in 'autogluon_model' directory.")
print(f"All outputs saved in '{output_dir}' and '{drive_dir}'.")

# 💾 Impute the missing values of Tempreture (1946-1969)

>


The model is


In [ ]:
data = dataAll
satellite_mean = data.loc['1981-01-01':'2024-12-31']['SatelliteAverageMinTemperature'].mean()
print(f"Mean SatelliteAverageMinTemperature (1981–2024): {satellite_mean:.2f}")

Mean SatelliteAverageMinTemperature (1981–2024): 17.08


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
from sklearn.metrics import mean_squared_error
from glob import glob

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# File paths
url = 'https://github.com/mtofighi/ChilwaBasin/blob/main/ChilwaBasin_DataAnalysis_032024/Dataset/ChilwaBasin_Dataset_09252025.xlsx?raw=true'
output_dir = '/content/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250925'
drive_dir = '/content/drive/My Drive/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250905'


gee_folder = '/content/drive/My Drive/Malawi/GEE_Folder'

era5_file = f'{gee_folder}/ChilwaBasin_ERA5Land_Temperature_Daily_1950_2025.csv'
imputed_file = f'{output_dir}/imputed_SatelliteAverageMinTemperature_1946_2025.csv'
drive_imputed_file = f'{drive_dir}/imputed_SatelliteAverageMinTemperature_1946_2025.csv'
plot_file = f'{output_dir}/imputed_plot.png'
drive_plot_file = f'{drive_dir}/imputed_plot.png'
updated_dataset_file = f'{output_dir}/ChilwaBasin_Dataset_Updated_1946_2025.xlsx'
drive_updated_dataset_file = f'{drive_dir}/ChilwaBasin_Dataset_Updated_1946_2025.xlsx'

# Define temperature column
temp_column = 'SatelliteAverageMinTemperature'

# Create directories
os.makedirs(output_dir, exist_ok=True)
os.makedirs(drive_dir, exist_ok=True)
print(f'Output directories created: {output_dir}, {drive_dir}')

# Load dataset directly from Excel
print('Loading dataset from Excel...')
try:
    dataAll = pd.read_excel(url, sheet_name='ChilwaBasinMonthlyDataset')
except Exception as e:
    print(f'Error loading Excel file: {e}')
    raise
print(f'Available columns in dataAll: {list(dataAll.columns)}')

# Verify dataset
if 'Date' not in dataAll.columns:
    print('Error: "Date" column not found in dataAll.')
    raise KeyError('Missing Date column in dataAll.')
if temp_column not in dataAll.columns:
    print(f'Error: "{temp_column}" column not found in dataAll. Available columns: {list(dataAll.columns)}')
    raise KeyError(f'Missing {temp_column} column in dataAll.')
if not pd.api.types.is_datetime64_any_dtype(dataAll['Date']):
    print('Converting "Date" column to datetime...')
    dataAll['Date'] = pd.to_datetime(dataAll['Date'], errors='coerce')
    if dataAll['Date'].isna().any():
        print('Error: Invalid dates in Date column after conversion.')
        raise ValueError('Invalid dates in dataAll.')
dataAll = dataAll.set_index('Date').sort_index()
if dataAll.index.has_duplicates:
    print('Warning: Duplicate dates found in dataAll. Keeping first occurrence.')
    dataAll = dataAll[~dataAll.index.duplicated(keep='first')]
print(f'Original dataset loaded. Shape: {dataAll.shape}')
print(f'Original data range: {dataAll.index.min()} to {dataAll.index.max()}')
missing_original = dataAll[temp_column].isna().sum()
print(f'Missing {temp_column} values in original data: {missing_original}')
if missing_original > 0:
    print(f'Warning: {missing_original} missing values detected in original {temp_column}.')

# Load ERA5-Land data
print('Loading ERA5-Land data...')
if os.path.exists(era5_file):
    era5_data = pd.read_csv(era5_file, parse_dates=['Date'], date_format='%m/%d/%Y')
else:
    print(f'Warning: {era5_file} not found. Attempting to load yearly CSVs...')
    era5_files = glob(f'{gee_folder}/ChilwaBasin_ERA5Land_Temperature_Daily_*.csv')
    if not era5_files:
        print(f'Error: No ERA5-Land CSV files found in {gee_folder}.')
        raise FileNotFoundError
    era5_data = pd.concat([pd.read_csv(f, parse_dates=['Date'], date_format='%m/%d/%Y') for f in era5_files])
try:
    era5_data.set_index('Date', inplace=True)
except pd.errors.ParserError:
    print('Error: Invalid date format in ERA5-Land CSV. Expected MM/dd/YYYY.')
    raise
era5_data = era5_data.sort_index()
if era5_data.index.has_duplicates:
    print('Warning: Duplicate dates found in ERA5-Land data. Keeping first occurrence.')
    era5_data = era5_data[~era5_data.index.duplicated(keep='first')]
if 'SatelliteAverageMinTemperature' not in era5_data.columns:
    print('Error: ERA5-Land CSV must contain "SatelliteAverageMinTemperature" column.')
    raise KeyError('Missing SatelliteAverageMinTemperature in ERA5-Land data.')
missing_era5 = era5_data['SatelliteAverageMinTemperature'].isna().sum()
print(f'ERA5-Land data loaded. Shape: {era5_data.shape}')
print(f'ERA5-Land data range: {era5_data.index.min()} to {era5_data.index.max()}')
print(f'Missing SatelliteAverageMinTemperature values in ERA5-Land data: {missing_era5}')
if missing_era5 > 0:
    print(f'Warning: {missing_era5} missing values detected in ERA5-Land SatelliteAverageMinTemperature.')

# Check date consistency
expected_dates = pd.date_range(start='1950-01-01', end='2025-08-05', freq='D')
if not era5_data.index.isin(expected_dates).all():
    print('Warning: ERA5-Land data contains unexpected dates.')
if era5_data.index.min() > pd.Timestamp('1950-01-01') or era5_data.index.max() < pd.Timestamp('2025-08-05'):
    print('Warning: ERA5-Land data does not cover full range (1950-01-01 to 2025-08-05).')

# Aggregate ERA5-Land to monthly data
print('Aggregating ERA5-Land data to monthly...')
era5_monthly = era5_data['SatelliteAverageMinTemperature'].resample('ME').mean()
print(f'ERA5-Land monthly data created. Shape: {era5_monthly.shape}')

# Create full monthly date range (1946–2025)
full_dates = pd.date_range(start='1946-01-31', end='2025-08-31', freq='ME')
imputed_data = pd.DataFrame(index=full_dates)

# Merge ERA5-Land monthly data
imputed_data['SatelliteAverageMinTemperature'] = era5_monthly
print(f'Merged ERA5-Land data for 1950–2025.')

# Preserve original TerraClimate data (1981–2024) where available
overlap_period = imputed_data.loc['1958-01-01':'2024-12-31'].index.intersection(dataAll.index)
imputed_data.loc[overlap_period, 'SatelliteAverageMinTemperature'] = dataAll.loc[overlap_period, temp_column]
print(f'Preserved original TerraClimate data for 1981–2024 where available.')

# Interpolate for 1946–1958
print('Interpolating SatelliteAverageMinTemperature for 1946–1958...')
imputed_data['SatelliteAverageMinTemperature'] = imputed_data['SatelliteAverageMinTemperature'].interpolate(method='linear', limit_direction='both')
missing_imputed = imputed_data['SatelliteAverageMinTemperature'].isna().sum()
print('Interpolation completed.')
if missing_imputed > 0:
    print(f'Error: {missing_imputed} missing values remain after interpolation.')
    raise ValueError('Interpolation failed to fill all missing values.')
print(f'Imputed data range: {imputed_data.index.min()} to {imputed_data.index.max()}')

# Save imputed temperature data
imputed_data.to_csv(imputed_file)
shutil.copy(imputed_file, drive_imputed_file)
print(f'Imputed SatelliteAverageMinTemperature (1946–2025) saved as {imputed_file} and copied to Google Drive.')

# Update original dataset
print('Updating original dataset...')
updated_data = dataAll.copy()
updated_data['SatelliteAverageMinTemperature'] = imputed_data['SatelliteAverageMinTemperature'].reindex(updated_data.index, method='ffill')
missing_updated = updated_data['SatelliteAverageMinTemperature'].isna().sum()
print('Merged imputed SatelliteAverageMinTemperature.')
if missing_updated > 0:
    print(f'Warning: {missing_updated} missing values in updated dataset SatelliteAverageMinTemperature.')
updated_data.to_excel(updated_dataset_file)
shutil.copy(updated_dataset_file, drive_updated_dataset_file)
print(f'Updated dataset saved as {updated_dataset_file} and copied to Google Drive.')

# Validate imputation on 1958–2024
print('Validating imputation on 1958–2024...')
validation_data = dataAll.loc['1958-01-01':'2024-12-31', temp_column]
imputed_validation = imputed_data.loc['1981-01-01':'2024-12-31', 'SatelliteAverageMinTemperature']
common_dates = validation_data.index.intersection(imputed_validation.index)
validation_data = validation_data.loc[common_dates].dropna()
imputed_validation = imputed_validation.loc[common_dates].dropna()
if len(validation_data) > 0:
    rmse = np.sqrt(mean_squared_error(validation_data, imputed_validation))
    print(f'Validation RMSE (1958–2024): {rmse:.2f}°C')
    if rmse > 2.0:
        print('Warning: High RMSE (>2.0°C) suggests potential misalignment. Check ERA5-Land data or calibration.')
else:
    print('Warning: No overlapping data for validation (1981–2024).')

# Visualize imputed data
print('Generating imputation plot...')
plt.figure(figsize=(10, 6))
plt.plot(imputed_data.loc['1946-01-01':'1957-12-31'].index,
         imputed_data.loc['1946-01-01':'1957-12-31']['SatelliteAverageMinTemperature'],
         label='Imputed (1946–1969)', color='red')
plt.plot(imputed_data.loc['1959-01-01':'2025-08-31'].index,
         imputed_data.loc['1959-01-01':'2025-08-31']['SatelliteAverageMinTemperature'],
         label='ERA5-Land/TerraClimate (1958–2025)', color='blue')
plt.plot(dataAll.loc['1958-01-01':'2024-12-31'].index,
         dataAll.loc['1958-01-01':'2024-12-31', temp_column],
         label='Original TerraClimate (1981–2024)', color='green', alpha=0.5)
plt.title('Imputed SatelliteAverageMinTemperature (1946–2025)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.tight_layout()
plt.savefig(plot_file, dpi=300)
shutil.copy(plot_file, drive_plot_file)
plt.close()
print(f'Imputation plot saved as {plot_file} and copied to Google Drive.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directories created: /content/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250925, /content/drive/My Drive/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250905
Loading dataset from Excel...
Available columns in dataAll: ['Date', 'Month', 'Season', 'SatelliteAverageMinTemperature', 'SatelliteAverageMinTemperatureStandardizedAnomaly', 'SatelliteAverageMaxTemperature', 'AverageMeanTemperature', 'AverageMeanTemperatureAnomaly', 'AverageMeanTemperatureStandardizedAnomaly', 'ChancoMeanTemperature', 'ChingaleMeanTemperature', 'MakokaMeanTemperature', 'NaminjiwaMeanTemperature', 'NtajaMeanTemperature', 'ZombaRTCMeanTemperature', 'AverageMinTemperature', 'AverageMinTemperatureAnomaly', 'AverageMinTemperatureStandardizedAnomaly', 'ChancoMinTemperature', 'ChingaleMinTemperature', 'MakokaMinTemperature', 'NaminjiwaMinTemperature', 'N

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directories created: /content/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250905, /content/drive/My Drive/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250905
Loading dataset from Excel...
Available columns in dataAll:
Original dataset loaded. Shape: (948, 130)
Original data range: 1946-01-01 00:00:00 to 2024-12-01 00:00:00
Missing SatelliteAverageMinTemperature values in original data: 420
Warning: 420 missing values detected in original SatelliteAverageMinTemperature.
Loading ERA5-Land data...
ERA5-Land data loaded. Shape: (27609, 1)
ERA5-Land data range: 1950-01-01 00:00:00 to 2025-08-03 00:00:00
Missing SatelliteAverageMinTemperature values in ERA5-Land data: 0
Warning: ERA5-Land data does not cover full range (1950-01-01 to 2025-08-05).
Aggregating ERA5-Land data to monthly...
ERA5-Land monthly data created. Shape: (908,)
Merged ERA5-Land data for 1950–2025.
Preserved original TerraClimate data for 1981–2024 where available.
Interpolating SatelliteAverageMinTemperature for 1946–1969...
Interpolation completed.
Imputed data range: 1946-01-31 00:00:00 to 2025-08-31 00:00:00
Imputed SatelliteAverageMinTemperature (1946–2025) saved as /content/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250905/imputed_SatelliteAverageMinTemperature_1946_2025.csv and copied to Google Drive.
Updating original dataset...
Merged imputed SatelliteAverageMinTemperature.
Warning: 1 missing values in updated dataset SatelliteAverageMinTemperature.
Updated dataset saved as /content/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250905/ChilwaBasin_Dataset_Updated_1946_2025.xlsx and copied to Google Drive.
Validating imputation on 1981–2024...
Warning: No overlapping data for validation (1981–2024).
Generating imputation plot...
Imputation plot saved as /content/Malawi/ChilwaRegression2025/SatelliteAverageMinTemperature_20250905/imputed_plot.png and copied to Google Drive.

# 🌧️ Rainfall Return Period Analysis – Chilwa Basin

## 📌 Objective
To analyze monthly satellite rainfall data from **January 1946 to December 2024** and compute the **empirical return period** for each month. This helps identify how rare or frequent each rainfall event is.

---

## 📊 Methodology

### 1. **Data Source**
- **File**: `ChilwaBasin_Dataset_09232025.xlsx`
- **Sheet**: `ChilwaBasinMonthlyDataset`
- **Columns Used**:
  - `Date`
  - `SatelliteAverageRainfall`

### 2. **Return Period Calculation**
- Rainfall values are sorted in **descending order**.
- Each value is ranked.
- Return period is calculated using the formula:

  \[
  \text{ReturnPeriod} = \frac{N + 1}{\text{Rank}}
  \]

  where:
  - \( N \) = total number of valid monthly rainfall records
  - \( \text{Rank} \) = position in sorted list

### 3. **Filtering**
- Rainfall values with return periods **less than 2 years** are considered insignificant and set to **0**.

### 4. **Standard Return Periods**
We use the following standard return intervals for comparison:


In [ ]:
# Install required packages
!pip install openpyxl

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Load dataset
url = 'https://raw.githubusercontent.com/mtofighi/ChilwaBasin/main/ChilwaBasin_DataAnalysis_032024/Dataset/ChilwaBasin_Dataset_09232025.xlsx'
dataAll = pd.read_excel(url, sheet_name='ChilwaBasinMonthlyDataset', engine='openpyxl')

# Extract relevant columns and drop missing values
rainfall_data = dataAll[['Date', 'SatelliteAverageRainfall']].dropna()

# Sort by descending rainfall to compute empirical return periods
rainfall_sorted = rainfall_data.sort_values(by='SatelliteAverageRainfall', ascending=False).reset_index(drop=True)
rainfall_sorted['Rank'] = rainfall_sorted.index + 1
N = len(rainfall_sorted)
rainfall_sorted['ReturnPeriod'] = (N + 1) / rainfall_sorted['Rank']

# Replace return periods less than 2 years with zero
rainfall_sorted['ReturnPeriod'] = rainfall_sorted['ReturnPeriod'].apply(lambda x: x if x >= 2 else 0)

# Define standard return periods
standard_periods = [2, 5, 10, 25, 50, 100, 200, 500]

# Match each rainfall to the closest standard return period
def closest_period(rp):
    if rp == 0:
        return 0
    return min(standard_periods, key=lambda x: abs(x - rp))

rainfall_sorted['ClosestStandardPeriod'] = rainfall_sorted['ReturnPeriod'].apply(closest_period)

# Merge back with original data using Date as key to avoid duplicates
rainfall_combined = pd.merge(rainfall_data, rainfall_sorted[['SatelliteAverageRainfall', 'ReturnPeriod', 'ClosestStandardPeriod']],
                             on='SatelliteAverageRainfall', how='left')

# Drop duplicates and sort by ascending date
rainfall_combined = rainfall_combined.drop_duplicates(subset='Date').sort_values(by='Date').reset_index(drop=True)

# Print rainfall amounts that correspond to exact standard return periods
print("Rainfall values with exact standard return periods:")
for period in standard_periods:
    match = rainfall_sorted[np.isclose(rainfall_sorted['ReturnPeriod'], period, atol=0.5)]
    if not match.empty:
        value = match.iloc[0]
        print(f"Return Period {period} years: Rainfall = {value['SatelliteAverageRainfall']:.2f} mm on {value['Date'].strftime('%Y-%m')}")
    else:
        print(f"Return Period {period} years: No exact match found")

# Total number of records
N = len(rainfall_sorted)

# Print rainfall values based on empirical return periods
print("Rainfall values based on empirical return periods:")
for T in standard_periods:
    rank = int(round((N + 1) / T))
    if rank <= N:
        value = rainfall_sorted.iloc[rank - 1]
        print(f"Return Period {T} years: Rainfall = {value['SatelliteAverageRainfall']:.2f} mm on {value['Date'].strftime('%Y-%m')}")
    else:
        print(f"Return Period {T} years: Rank {rank} exceeds dataset size")



# Save results to Excel
target = 'SatelliteRainfallReturnPeriod'
date_str = datetime.now().strftime('%Y%m%d')
output_dir = f'/content/Malawi/ChilwaRegression2025/{target}_{date_str}'
drive_dir = f'/content/drive/My Drive/Malawi/ChilwaRegression2025/{target}_{date_str}'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(drive_dir, exist_ok=True)

output_file = f'{output_dir}/Rainfall_ReturnPeriods_{date_str}.xlsx'
drive_file = f'{drive_dir}/Rainfall_ReturnPeriods_{date_str}.xlsx'
rainfall_combined.to_excel(output_file, index=False)
rainfall_combined.to_excel(drive_file, index=False)

print(f"\n✅ Results saved to:\n- Colab: {output_file}\n- Google Drive: {drive_file}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rainfall values with exact standard return periods:
Return Period 2 years: Rainfall = 73.50 mm on 1990-11
Return Period 5 years: Rainfall = 212.80 mm on 1965-12
Return Period 10 years: Rainfall = 266.50 mm on 1974-02
Return Period 25 years: Rainfall = 299.00 mm on 2019-03
Return Period 50 years: Rainfall = 331.10 mm on 1972-01
Return Period 100 years: No exact match found
Return Period 200 years: No exact match found
Return Period 500 years: No exact match found
Rainfall values based on empirical return periods:
Return Period 2 years: Rainfall = 40.70 mm on 2000-10
Return Period 5 years: Rainfall = 197.60 mm on 2001-01
Return Period 10 years: Rainfall = 266.50 mm on 1975-02
Return Period 25 years: Rainfall = 299.00 mm on 2019-03
Return Period 50 years: Rainfall = 331.10 mm on 1972-01
Return Period 100 years: Rainfall = 353.60 mm on 1986-01
Return Period 200 y